In [33]:
import torch
import numpy as np

from dinosaw.utils import get_features, add_custom_font
from dinosaw.wrappers import ModelTypes, MODEL_NAMES, get_models


from skimage.color import label2rgb

from sklearn.cluster import KMeans
from sklearn.preprocessing import scale
from PIL import Image
from PIL.ImageColor import getcolor

import matplotlib.pyplot as plt

SEED = 100001
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

In [34]:
enabled_models: tuple[ModelTypes, ...] = ('dinov2_s', 'dvt_dinov2_s', 'alibi_coco_dinov2_s', 'dinov3_s+', 'alibi_dinov3_s+_norm_wrap_ms')
models = get_models(enabled_models, DEVICE, True, "../../models/checkpoints", "../../models/dinov3")

2026-07-24 10:21:00 | I | factory.py                 : 152 | Building wrapper 'dinov2_s' on device cuda:0
2026-07-24 10:21:00 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)


2026-07-24 10:21:01 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-24 10:21:01 | I | factory.py                 : 152 | Building wrapper 'dvt_dinov2_s' on device cuda:0
2026-07-24 10:21:01 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)
2026-07-24 10:21:01 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-24 10:21:01 | I | factory.py                 : 152 | Building wrapper 'alibi_coco_dinov2_s' on

In [35]:
image_fnames = ['diff_shapes_518.png',  'wuppertal.jpg', '000004.jpg']
imgs: list[Image.Image] = []
ks = [4, 3, 5, 3]
results: dict[ModelTypes, list[np.ndarray]] = {mt: [] for mt in enabled_models}

In [36]:
for img_fname in image_fnames:
    img = Image.open(f'../paper_figures/data/PCA/{img_fname}').convert('RGB')
    img = img.resize((518, 518))
    imgs.append(img)

In [37]:
# %%capture
for key, model in models.items():
    for i, img in enumerate(imgs):
        feats_li = []
        feats = get_features(model, img, True)
        h, w, c = feats.shape
        flat = feats.reshape((h * w, c))
        flat = scale(flat)
        reduced = KMeans(ks[i], n_init=15).fit_predict(flat)
        reduced_2D = reduced.reshape((h, w))
        results[key].append(reduced_2D)

2026-07-24 10:21:02 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 10:21:02 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-07-24 10:21:02 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 10:21:02 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-07-24 10:21:02 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 10:21:02 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:275: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:294: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:275: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:294: UserWarning: Numerical issues were encountered when scaling the data and m

2026-07-24 10:21:02 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 10:21:02 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:275: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:294: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:275: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:294: UserWarning: Numerical issues were encountered when scaling the data and m

2026-07-24 10:21:02 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 10:21:02 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-07-24 10:21:02 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 10:21:02 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]


...

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:275: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:294: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:275: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:294: UserWarning: Numerical issues were encountered when scaling the data and m

2026-07-24 10:21:03 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 10:21:03 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,512,512] -> f: [1,384,32,32]
2026-07-24 10:21:03 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 10:21:03 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,512,512] -> f: [1,384,32,32]
2026-07-24 10:21:03 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 10:21:03 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,512,512] -> f: [1,384,32,32]


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:275: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:294: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:275: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:294: UserWarning: Numerical issues were encountered when scaling the data and m

2026-07-24 10:21:03 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 10:21:03 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,512,512] -> f: [1,384,32,32]
2026-07-24 10:21:03 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 10:21:03 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,512,512] -> f: [1,384,32,32]


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:275: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:294: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:275: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:294: UserWarning: Numerical issues were encountered when scaling the data and m

In [38]:
# %%capture
plt.style.use("thesis.mplstyle")#
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')


COLOURS = [
    "#648FFF",
    "#785EF0",
    "#DC267F",
    "#FE6100",
    "#FFB000"
]
COLORS = [[v / 255.0 for v in getcolor(c, "RGB")] for c in COLOURS]


W, H = 2.25, 2.5 * 2.3

N_COLS = len(enabled_models) + 1 
N_ROWS = len(image_fnames)

fig, axs = plt.subplots(N_COLS, N_ROWS, figsize=(W, H))

for i, img in enumerate(imgs):
    ax = axs[0, i]
    ax.imshow(img)
    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_title(r'$k$' + rf"={ks[i]}")

    for j, key in enumerate(enabled_models):
        ax = axs[j + 1, i]
        selected_ch = results[key][i]
        remapped = label2rgb(selected_ch, colors=COLORS, bg_label=-1)
        ax.imshow(remapped, cmap='tab10', interpolation='nearest')
        ax.set_xticks([])
        ax.set_yticks([])            

        if i == 0:
            model_name = MODEL_NAMES[key]
            model_name = model_name.replace('(COCO)', '')
            weight = 700 if 'alibi' in model_name.lower() else 500
            axs[ j+ 1, 0,].set_ylabel(model_name, weight=weight)


SAVE = True
if SAVE:
    plt.savefig("saved/08.pdf", dpi=300, bbox_inches='tight', pad_inches=0.05)
    plt.close()

findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight 500, now using 300.
